In [170]:
import requests
import pandas as pd
from datetime import datetime

TOKEN = "y0__xDe_Im4BBiirjkg2Zzs_xP3srDDcLIeER6ns2I71_wDiFmYXg"
COUNTER_ID = 103580753

path = 'tmp/'
city = ('Moscow','Saint Petersburg')
dimensions = ['ym:s:date', 'ym:s:browser']

url = 'https://api-metrika.yandex.net/stat/v1/data/'

headers = {
    'Authorization': f"OAuth {TOKEN}"
}


metrics = [
    'ym:s:visits',
    'ym:s:users'
]

params = {
    'metrics': ','.join(metrics),
    'date1': '2025-12-24',
    'date2': '2025-12-31',
    'ids': COUNTER_ID,
    'dimensions': ','.join(dimensions),
    'filters': f'ym:s:regionCityName=.{city}'
}


try:
    response = requests.get(url, params=params, headers=headers)
    response.raise_for_status()
    data = response.json()

    columns = []
    dimensions_name = data['query']['dimensions']
    metrics_name = data['query']['metrics']
    strip_names = dimensions_name + metrics_name
    for name in strip_names:
        columns.append(name[5:])
    columns.append('update_at')

    rows = []
    for row in data['data']:
        dimension_values = [dim['name'] for dim in row['dimensions']]
        metrics = row['metrics']
        now = datetime.now()
        now_str = now.strftime("%Y-%m-%d %H:%M:%S")
        rows.append(dimension_values + metrics + [now_str])

    df = pd.DataFrame(rows, columns=columns)
    df_sorted = df.sort_values(['date', 'browser'])
    df_sorted.to_csv('tmp/usersAndVisitsByRegion.csv', index=False)
except EOFError as e:
    print(f'Ошибка: {e}')